# 05 — Customer Sentiment Analysis

**Goal:** for every call, the **customer's** sentiment (positive / neutral / negative) with a short, evidence-based justification, plus a comparison of the beginning and the end of the call.

**Runs on:** a GPU notebook: Google Colab (T4) or Kaggle (Settings → Accelerator: GPU, Internet: on).

**Input:** `results/segmented_turns.csv` — our Whisper transcripts with speaker roles (notebook 03).

**Output**
- `results/sentiment.csv` — one row per call: label, justification, evidence, beginning vs end
- `results/sentiment_turns.csv` — classifier scores for every customer turn
- `results/sentiment_review.csv` — 10 calls prepared for manual review
- `results/sentiment_run_info.json` — models and library versions

**Sections**
1. Key ideas
2. Setup
3. Load the models
4. Context and negation: a quick test
5. Method 1 — classifier on every customer turn
6. Method 2 — language model on the whole call
7. Comparing the methods and the dataset labels
8. Disagreements
9. Manual review sheet
10. Findings
11. Save and download (Colab)

## 1. Key ideas

**Customer only.** The agent is trained to sound positive in every call, so including the agent would hide the customer's feelings. We analyse the customer's turns only.

**Sentiment is not satisfaction.** Sentiment is the emotion a person *expresses* ("thank you so much!" → positive). Satisfaction is whether their need was *met*. A customer can stay neutral and still be fully served, or sound happy before the problem is solved.

**Context and negation.** Word-based models can be fooled: in "no, that's all, thank you" the word *no* answers a question and is not negative. A model that reads the whole conversation handles this better.

**Two methods, used together**

| | Method 1: classifier | Method 2: language model |
|---|---|---|
| Model | `cardiffnlp/twitter-roberta-base-sentiment-latest` (RoBERTa, trained on ~124M tweets, CC-BY-4.0) | `Qwen3-4B-Instruct-2507` (same as notebook 04) |
| Input | One customer turn at a time | The whole call |
| Output | Probabilities for 3 classes | One label + justification + quoted evidence |
| Strength | Consistent scores; shows how sentiment changes during the call | Uses context; explains its decision |
| Weakness | Sees no context; trained on tweets, not calls | Can be wrong with confidence; needs checking |

**Final label = Method 2** (it uses context and gives a reason). Method 1 is a second opinion and gives the **beginning-vs-end** comparison.

**Expect mostly neutral.** In notebook 01, 67% of customer utterances had a neutral dataset label and only 3.4% negative. So accuracy alone would be misleading: a model that always says "neutral" would look good.

## 2. Setup

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")       # Kaggle notebooks: an alternative free GPU
REPO_URL = "https://github.com/ahmedalmoutiry-wq/call-center-agent-performance-analysis.git"
REPO_DIR = ("/content" if IN_COLAB else "/kaggle/working") + "/call-center-agent-performance-analysis"

if IN_COLAB or IN_KAGGLE:
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    os.chdir(REPO_DIR)
    !pip install -q num2words==0.5.14
    !python scripts/download_data.py          # metadata + transcripts (for the dataset's emotion labels)

In [ ]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import RESULTS_DIR, build_utterances_table
from src.llm import DEFAULT_MODELS, load_llm
from src.sentiment import (CLASSIFIER_ID, LABELS, build_messages, evidence_is_customer_quote,
                           label_from_probabilities, load_classifier, parse_sentiment)
from src.summarize import transcript_text

pd.set_option("display.max_colwidth", 120)
ORDER = {"negative": 0, "neutral": 1, "positive": 2}

turns = pd.read_csv(RESULTS_DIR / "segmented_turns.csv")
subset = pd.read_csv(RESULTS_DIR / "subset_ids.csv")
customer_turns = turns[turns["role"] == "Customer"].copy()
print(f"Calls: {turns['call_id'].nunique()}  |  customer turns: {len(customer_turns)}")

## 3. Load the models

In [ ]:
classify = load_classifier(CLASSIFIER_ID)
generate, LLM_ID = load_llm(DEFAULT_MODELS)

import torch, transformers
run_info = {"classifier": CLASSIFIER_ID, "llm": LLM_ID, "transformers": transformers.__version__,
            "torch": torch.__version__, "decoding": "greedy (do_sample=False)",
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}
run_info

## 4. Context and negation: a quick test

Before using the classifier, we check how it handles typical call phrases. Each sentence is judged **alone**, without the conversation.

In [ ]:
probe = ["no that's all thank you", "no problem", "nope that will be everything",
         "i lost my debit card", "i'm not happy with this at all", "great thank you so much"]
classify(probe).assign(sentence=probe).round(2)[["sentence", "label"] + [f"p_{l}" for l in LABELS]]

## 5. Method 1 — classifier on every customer turn

Every customer turn gets three probabilities. The **call label** is the class with the highest **average** probability.

**Beginning vs end:** we compare the customer's first and last turns that have at least 3 words (one-word turns like "yes" carry little emotion).

In [ ]:
scores = classify(customer_turns["text"].astype(str).tolist())
customer_turns = pd.concat([customer_turns.reset_index(drop=True), scores], axis=1)
customer_turns.to_csv(RESULTS_DIR / "sentiment_turns.csv", index=False)

def start_end(call_turns):
    long_turns = call_turns[call_turns["n_words"] >= 3].sort_values("turn_index")
    if long_turns.empty:
        return pd.Series({"start_label": None, "end_label": None})
    return pd.Series({"start_label": long_turns["label"].iloc[0], "end_label": long_turns["label"].iloc[-1]})

classifier_calls = customer_turns.groupby("call_id").apply(
    lambda g: pd.concat([pd.Series({"classifier_label": label_from_probabilities(g)}), start_end(g)]),
    include_groups=False).reset_index()

def change(row):
    if not row["start_label"] or not row["end_label"]:
        return "unknown"
    diff = ORDER[row["end_label"]] - ORDER[row["start_label"]]
    return "improved" if diff > 0 else "worsened" if diff < 0 else "same"

classifier_calls["start_to_end"] = classifier_calls.apply(change, axis=1)
print("Call labels (classifier):", classifier_calls["classifier_label"].value_counts().to_dict())
print("Beginning -> end:", classifier_calls["start_to_end"].value_counts().to_dict())
classifier_calls.head()

## 6. Method 2 — language model on the whole call

The model reads the full conversation and returns a label, a one-sentence justification and a **quote** of the customer's words as evidence. We then check that the quote is real: it must appear in what the **customer** said (not the agent).

In [ ]:
transcripts = {c: transcript_text(t) for c, t in turns.groupby("call_id")}
customer_text = customer_turns.groupby("call_id")["text"].apply(" ".join).to_dict()

rows = []
for call_id, transcript in transcripts.items():
    raw = generate(build_messages(transcript))
    result = parse_sentiment(raw)
    result["evidence_is_customer_quote"] = evidence_is_customer_quote(result["evidence"], customer_text.get(call_id, ""))
    rows.append({"call_id": call_id, **result, "raw_output": raw})

llm_calls = pd.DataFrame(rows)
print(f"Format OK: {llm_calls['parse_ok'].sum()} of {len(llm_calls)}")
print(f"Evidence is a real customer quote: {llm_calls['evidence_is_customer_quote'].sum()} of {len(llm_calls)}")
print("Call labels (language model):", llm_calls["sentiment"].value_counts().to_dict())
llm_calls[["call_id", "sentiment", "justification", "evidence"]].head(8)

## 7. Comparing the methods and the dataset labels

There are no human sentiment labels in the dataset. The only reference is the dataset's own **emotion labels**, which were produced by a commercial model (Gridspace) on the **audio and reference text**. So "agreement" here means agreement between models, **not accuracy**.

For the dataset reference we average the emotion probabilities of the customer's utterances (with words) in each call.

In [ ]:
reference = build_utterances_table(subset["call_id"].tolist())
ref_customer = reference[(reference["speaker_role"] == "caller") & ~reference["is_empty"] & ~reference["is_noise_only"]]
dataset_calls = ref_customer.groupby("call_id").apply(
    label_from_probabilities, include_groups=False
).rename("dataset_label").reset_index()

sentiment = (llm_calls.merge(classifier_calls, on="call_id")
                      .merge(dataset_calls, on="call_id")
                      .merge(subset[["call_id", "task_type"]], on="call_id"))

agreement = pd.Series({
    "language model = classifier": (sentiment["sentiment"] == sentiment["classifier_label"]).mean(),
    "language model = dataset label": (sentiment["sentiment"] == sentiment["dataset_label"]).mean(),
    "classifier = dataset label": (sentiment["classifier_label"] == sentiment["dataset_label"]).mean(),
}).round(2)
print(agreement.to_string(), "\n")
pd.crosstab(sentiment["sentiment"], sentiment["dataset_label"],
            rownames=["language model"], colnames=["dataset label"])

## 8. Disagreements

Calls where the language model and the classifier disagree are the most informative ones to read.

In [ ]:
disagree = sentiment[sentiment["sentiment"] != sentiment["classifier_label"]]
disagree[["call_id", "sentiment", "classifier_label", "dataset_label", "justification", "evidence"]]

## 9. Manual review sheet

The same 10 calls as the summary review (notebook 04), so one reading covers both. Fill in after reading the customer's words:
- `label_correct` — `yes` / `no`: is the language-model label right?
- `justification_supported` — `yes` / `no`: does the evidence support the reason?
- `review_notes` — what is wrong, if anything

In [ ]:
summary_review_file = RESULTS_DIR / "summary_review.csv"
review_ids = (pd.read_csv(summary_review_file)["call_id"].tolist() if summary_review_file.exists()
              else sentiment["call_id"].head(10).tolist())

review = sentiment[sentiment["call_id"].isin(review_ids)][
    ["call_id", "sentiment", "justification", "evidence", "classifier_label", "start_to_end"]].copy()
review.insert(1, "customer_words", review["call_id"].map(customer_text))
review["label_correct"], review["justification_supported"], review["review_notes"] = "", "", ""
review.to_csv(RESULTS_DIR / "sentiment_review.csv", index=False)
print(f"Review sheet: {len(review)} calls -> results/sentiment_review.csv")

## 10. Findings

_To be written after the run, from the numbers above and the manual review._

## 11. Save and download (Colab)

In [ ]:
out_cols = ["call_id", "task_type", "sentiment", "justification", "evidence", "evidence_is_customer_quote",
            "start_label", "end_label", "start_to_end", "classifier_label", "dataset_label", "parse_ok", "raw_output"]
sentiment[out_cols].to_csv(RESULTS_DIR / "sentiment.csv", index=False)
(RESULTS_DIR / "sentiment_run_info.json").write_text(json.dumps(run_info, indent=2))

if IN_COLAB or IN_KAGGLE:
    import shutil
    export_dir = Path("/content/sentiment_results" if IN_COLAB else "/kaggle/working/sentiment_results")
    export_dir.mkdir(exist_ok=True)
    for name in ["sentiment.csv", "sentiment_turns.csv", "sentiment_review.csv", "sentiment_run_info.json"]:
        shutil.copy(RESULTS_DIR / name, export_dir / name)
    archive = shutil.make_archive(str(export_dir), "zip", export_dir)
    if IN_COLAB:
        from google.colab import files
        files.download(archive)
    else:
        print(f"Saved {archive}: download it from the Output panel on the right.")